## Connect and List Entries

Connect to your ReductStore instance and list the entries in a bucket.

In [119]:
import pandas as pd
from reduct import Client

REDUCT_URL = "https://orion.reductsoft.eu"
API_TOKEN = "ReductCool2"

async with Client(REDUCT_URL, api_token=API_TOKEN, verify_ssl=False) as client:
    bucket = await client.get_bucket("ros_data")
    entries = await bucket.get_entry_list()
    df = (
        pd.DataFrame([{"entry": e.name} for e in entries])
        .sort_values("entry")
        .reset_index(drop=True)
    )

df.head(100)

,entry
0,Pablo05/gps/rpy
1,Pablo05/odom
2,Pablo05/point
3,Pablo05/sensor/gps/fix
4,Pablo39/gps/rpy
5,Pablo39/odom
6,Pablo39/point
7,Pablo39/sensor/gps/fix
8,center_camera/camera_info
9,center_camera/image_color/compressed


## Extract ROS Data to DataFrame

Extract tabular data from the ROS messages and load it into a Pandas DataFrame for analysis.

In [120]:
import json
import pandas as pd

when = {"#ext": {"ros": {"extract": {}}}, "$limit": 10}


rows = []
async with Client(REDUCT_URL, api_token=API_TOKEN, verify_ssl=False) as client:
    bucket = await client.get_bucket("ros_data")

    async for rec in bucket.query("imu/acceleration", when=when):
        blob = await rec.read_all()
        rows.append(json.loads(blob))

df = pd.json_normalize(rows)
df.head()

,header.stamp.sec,header.stamp.nanosec,header.frame_id,vector.y,vector.x,vector.z
0,1773934634,65827948,imu_link,0.482852,-0.425704,9.909235
1,1773934634,80130789,imu_link,0.185370,-0.539503,9.864081
2,1773934634,80463940,imu_link,0.374574,-0.422291,9.805202
3,1773934634,236495327,imu_link,0.234185,-0.408010,9.843310
4,1773934634,244890547,imu_link,0.342491,-0.426396,9.870441


## Export to MCAP

Export the ROS messages to an MCAP file for use with other tools.

In [95]:
from reduct import Client
from datetime import datetime, timedelta, timezone

stop = datetime.now(timezone.utc)
start = stop - timedelta(seconds=5)

when = {
    "#ext": {
        "ros": {
            "export": {"format": "mcap"},
        },
    },
}

async with Client(REDUCT_URL, api_token=API_TOKEN, verify_ssl=False) as client:
    bucket = await client.get_bucket("ros_data")
    async for rec in bucket.query("*", start, stop, when=when):
        blob = await rec.read_all()
        with open("data.mcap", "wb") as f:
            f.write(blob)

## Vizualize in Foxglove

Create a sharable link to visualize the extracted data in Foxglove.

In [ ]:
from datetime import datetime, timedelta

stop = datetime.now(timezone.utc)
start = stop - timedelta(seconds=5)

when = {
    "#ext": {
        "ros": {
            "export": {"format": "mcap"},
        },
    },
}

expire_at = datetime.now(timezone.utc) + timedelta(days=30)
record_index = 0
file_name = f"data.mcap"

async with Client(REDUCT_URL, api_token=API_TOKEN, verify_ssl=False) as client:
    bucket = await client.get_bucket("ros_data")
    link = await bucket.create_query_link(
        "*",
        start,
        stop,
        expire_at=expire_at,
        record_index=record_index,
        file_name=file_name,
    )
    print(link)

https://orion.reductsoft.eu/api/v1/links/data.mcap?ct=yCCCYUX_0IqbjaGIlnLy7-Le6di4Zgz-SuNhkiE2BvoixxCDyF72TAz39STn6Q4MiqFsSMjb0OKfOrAKfyGam_pMngYzPXqA7Q98pzUh7LIASEwTfWyYpYdB25iM9_dnmFQw2FQrRXrGybjaGVdwUzA8G__pPCLxVe_RSUpoA_24FsLsvlgkPT-ECG8B1Cn4Hux6CCnc1pV609WKKS8lUJqkaoeZkv3Vnj34Yr_WEQehRMWwrTYzJd_AoLae-wqaOq00qPkGmYJTXr8k2igxtrVagT3Bngp5m0uuyxU2&s=Qcaw7W32BRnXWz1rjcUx_A&i=init-token&n=O_JdTJm--c4vroJpXnJOgQ&r=0
